# 13.1: Beam search

Because every additional token multiplies in a probability less than one, raw log-probability favors short outputs. Implementations therefore usually apply *length normalization*, for example dividing the score by $t^\alpha$ for some $\alpha$ between 0 and 1, before comparing finished hypotheses.


In [ ]:
import numpy as np

def beam_search(step_logprobs, bos, eos, beam_size=4, max_len=20, alpha=0.7):
    """step_logprobs(prefix) -> array of log-probabilities over the vocabulary."""
    beams = [([bos], 0.0)]
    finished = []
    for _ in range(max_len):
        candidates = []
        for prefix, score in beams:
            lp = step_logprobs(prefix)
            for tok in np.argsort(-lp)[:beam_size]:        # only the best few per beam
                candidates.append((prefix + [int(tok)], score + float(lp[tok])))
        candidates.sort(key=lambda c: c[1], reverse=True)
        beams = []
        for prefix, score in candidates:
            (finished if prefix[-1] == eos else beams).append((prefix, score))
            if len(beams) == beam_size:
                break
        if not beams:
            break
    finished.extend(beams)
    return max(finished, key=lambda c: c[1] / (len(c[0]) ** alpha))
